**Decision trees with a toy task and the UCI Adult dataset**

In [1]:
import collections
from io import StringIO

import numpy as np
import pandas as pd
import pydotplus  # pip install pydotplus
import seaborn as sns
from ipywidgets import Image
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, export_graphviz

from matplotlib import pyplot as plt
plt.rcParams["figure.figsize"] = (10, 8)

Part 1. Toy dataset “Will They? Won’t They?”

In [2]:
# creating dataframe using dummy variables

def create_df(dic, feature_list):
    out = pd.DataFrame(dic)
    out = pd.concat([out, pd.get_dummies(out[feature_list])], axis=1) # refer extranotes.ipynb
    out.drop(feature_list, axis=1, inplace=True)
    return out

In [3]:
# Some feature values are present in train and absent in test and vice-versa.
def intersect_features(train, test):
    common_feat = list(set(train.keys()) & set(test.keys()))  
    return train[common_feat], test[common_feat]

In [4]:
features = ["Looks", "Alcoholic_beverage", "Eloquence", "Money_spent"]

Training data

In [5]:
df_train = {}
df_train["Looks"] = [
    "handsome",
    "handsome",
    "handsome",
    "repulsive",
    "repulsive",
    "repulsive",
    "handsome",
]
df_train["Alcoholic_beverage"] = ["yes", "yes", "no", "no", "yes", "yes", "yes"]
df_train["Eloquence"] = ["high", "low", "average", "average", "low", "high", "average"]
df_train["Money_spent"] = ["lots", "little", "lots", "little", "lots", "lots", "lots"]
df_train["Will_go"] = LabelEncoder().fit_transform(["+", "-", "+", "-", "-", "+", "+"])

#print(df_train)

df_train = create_df(df_train, features)
df_train

,Will_go,Looks_handsome,Looks_repulsive,Alcoholic_beverage_no,Alcoholic_beverage_yes,Eloquence_average,Eloquence_high,Eloquence_low,Money_spent_little,Money_spent_lots
0,0,True,False,False,True,False,True,False,False,True
1,1,True,False,False,True,False,False,True,True,False
2,0,True,False,True,False,True,False,False,False,True
3,1,False,True,True,False,True,False,False,True,False
4,1,False,True,False,True,False,False,True,False,True
5,0,False,True,False,True,False,True,False,False,True
6,0,True,False,False,True,True,False,False,False,True


Test data

In [ ]:
df_test = {}
df_test["Looks"] = ["handsome", "handsome", "repulsive"]
df_test["Alcoholic_beverage"] = ["no", "yes", "yes"]
df_test["Eloquence"] = ["average", "high", "average"]
df_test["Money_spent"] = ["lots", "little", "lots"]
df_test = create_df(df_test, features)
df_test

In [ ]:
# Some feature values are present in train and absent in test and vice-versa.
y = df_train["Will_go"]
df_train, df_test = intersect_features(train=df_train, test=df_test)
df_train

In [ ]:
df_test

Problems

1. What is the entropy ***s0*** of the initial system? By system states, we mean values of the binary feature “Will_go” - 0 or 1 - two states in total.

In [ ]:
df_train["Will_go"].value_counts()

# s0 = -4/7 log2 4/7 - 3/7 log2 3/7  approx = 0.98522814

Will_go
0    4
1    3
Name: count, dtype: int64

2. Let’s split the data by the feature “Looks_handsome”. What is the entropy 
 of the left group - the one with “Looks_handsome”. What is the entropy 
 in the opposite group? What is the information gain (IG) if we consider such a split?

## Split on: $\text{Looks\_handsome}$

### Step 1: Parent Entropy

Target column `Will_go` values: $[0, 1, 0, 1, 1, 0, 0]$

Counts:
- $0 \rightarrow 4$
- $1 \rightarrow 3$

$$p(0) = \frac{4}{7}, \qquad p(1) = \frac{3}{7}$$

Parent entropy:

$$S_0 = -\frac{4}{7}\log_2\frac{4}{7} - \frac{3}{7}\log_2\frac{3}{7}$$

$$S_0 \approx 0.9852$$

---

### Step 2: Left Group Entropy

Left group (`Looks_handsome = True`): $[0, 1, 0, 0]$

Counts:
- $0 \rightarrow 3$
- $1 \rightarrow 1$

$$p(0) = \frac{3}{4}, \qquad p(1) = \frac{1}{4}$$

Entropy:

$$S_L = -\frac{3}{4}\log_2\frac{3}{4} - \frac{1}{4}\log_2\frac{1}{4}$$

$$S_L \approx 0.8113$$

---

### Step 3: Right Group Entropy

Right group (`Looks_handsome = False`): $[1, 1, 0]$

Counts:
- $1 \rightarrow 2$
- $0 \rightarrow 1$

$$p(1) = \frac{2}{3}, \qquad p(0) = \frac{1}{3}$$

Entropy:

$$S_R = -\frac{2}{3}\log_2\frac{2}{3} - \frac{1}{3}\log_2\frac{1}{3}$$

$$S_R \approx 0.9183$$

---

### Step 4: Weighted Entropy After Split

Sizes:
- Left group: $4$
- Right group: $3$
- Total: $7$

Weighted entropy:

$$S_{\text{split}} = \frac{4}{7} S_L + \frac{3}{7} S_R$$

$$= \frac{4}{7}(0.8113) + \frac{3}{7}(0.9183)$$

$$\approx 0.8571$$

---

### Step 5: Information Gain

$$IG = S_0 - S_{\text{split}}$$

$$IG = 0.9852 - 0.8571$$

$$IG \approx 0.1281$$

---

### Final Answers

- Left group entropy: $\boxed{0.8113}$
- Right group entropy: $\boxed{0.9183}$
- Information Gain: $\boxed{0.1281}$

Q. Train a decision tree using sklearn on the training data. You may choose any depth for the tree.
Additional: display the resulting tree using graphviz. You can use pydot or a web-service